# Synthetic Control Method (SCM)

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


**Business setting:** the program launched in **one** market (or one country, one flagship store). No single control market matches it well, but a *weighted blend* might. SCM (Abadie, Diamond & Hainmueller 2010) finds nonnegative weights summing to 1 over a **donor pool** of untreated units so that the weighted combination reproduces the treated unit's *pre-period* outcome path (and optionally covariates). The post-period gap between the treated unit and its synthetic twin is the effect estimate.

Why business people like it: the counterfactual is transparent ("Synthetic Chicago = 0.4 × Minneapolis + 0.35 × Detroit + 0.25 × Cleveland") and the result is a picture, not a coefficient.

## Estimation with `pysyncon`

In [ ]:
%pip install pysyncon
from pysyncon import Dataprep, Synth

mkt_long = (panel.groupby(["week", "market"], as_index=False)
                 .revenue.mean())

dataprep = Dataprep(
    foo=mkt_long,
    predictors=["revenue"],
    predictors_op="mean",
    time_predictors_prior=range(20, 60),     # pre-period for matching
    dependent="revenue",
    unit_variable="market",
    time_variable="week",
    treatment_identifier=0,                  # treated market
    controls_identifier=[4, 5, 6, 7, 8, 9, 10, 11],
    time_optimize_ssr=range(0, 60),
)
synth = Synth()
synth.fit(dataprep=dataprep)
print(synth.weights().round(3))              # donor weights
synth.path_plot(time_period=range(0, 104), treatment_time=60)
synth.gaps_plot(time_period=range(0, 104), treatment_time=60)

## Inference: placebo-in-space permutation

Classic SCM has one treated unit, so standard errors don't apply. Abadie's solution is permutation: fit a synthetic control to **every donor** as if it were treated, and compare the real gap to the placebo gaps.

In [ ]:
from pysyncon import PlaceboTest

placebo = PlaceboTest()
placebo.fit(dataprep=dataprep, scm=synth)
placebo.gaps_plot(time_period=range(0, 104), treatment_time=60,
                  exclude_units=None, mspe_threshold=5)
print(placebo.pvalue(treatment_time=60))     # permutation p-value

Standard practice: drop placebos with terrible pre-period fit (high pre-MSPE) before comparing, or rank units by the ratio post-MSPE / pre-MSPE; the treated unit should rank at or near the top if the effect is real.

## Diagnostics and variants

- **Pre-period fit is the license to interpret.** If the synthetic unit cannot track the treated unit pre-launch, stop.
- **No extrapolation:** weights are nonnegative and sum to 1, so the treated unit must lie inside the donor pool's convex hull. A flagship market that dwarfs every donor can't be synthesized — consider the **augmented SCM** (Ben-Michael, Feller & Rothstein 2021; `pysyncon`'s `AugSynth`) which adds a bias-correcting outcome model, or scale outcomes (revenue per store).
- **In-time placebo:** fit using only data before a fake earlier launch; the "effect" should be ≈ 0.
- **Backdating / leave-one-out:** drop the highest-weight donor and refit; conclusions shouldn't hinge on a single donor.
- **Don't cherry-pick the donor pool** after seeing results; pre-register the pool (even informally, in the analysis doc).